# GKX Missingness Diagnosis

This notebook diagnoses why many GKX features are sparse or all-NaN for run `run_20260215_112600`, with emphasis on proving whether the issue is OHLCV-derived features or missing fundamentals inputs.

In [ ]:
from pathlib import Path
import pandas as pd

RUN_TAG = "run_20260215_112600"
base = Path("data")
coverage_path = base / "processed" / RUN_TAG / "gkx_coverage_report.csv"
gkx_monthly_path = base / "processed" / RUN_TAG / "gkx_monthly.parquet"
final_path = base / "processed" / RUN_TAG / "final_dataset.parquet"
quarterly_path = base / "raw" / RUN_TAG / "fundamentals_quarterly" / "quarterly_fundamentals.parquet"

coverage = pd.read_csv(coverage_path)
gkx_monthly = pd.read_parquet(gkx_monthly_path)
final_df = pd.read_parquet(final_path)
quarterly_df = pd.read_parquet(quarterly_path)

print("Coverage rows:", len(coverage))
print("GKX monthly shape:", gkx_monthly.shape)
print("Final daily shape:", final_df.shape)
print("Quarterly fundamentals shape:", quarterly_df.shape)

In [ ]:
all_nan = coverage[coverage["rows_non_null"] == 0].copy()
low_cov = coverage.sort_values("coverage_pct").head(30).copy()

print("All-NaN GKX features:", len(all_nan))
display(all_nan[["gkx_name", "alias", "coverage_pct"]].sort_values("gkx_name"))

print("Lowest-coverage features:")
display(low_cov[["gkx_name", "coverage_pct", "rows_non_null"]])

In [ ]:
# OHLCV-driven features should generally have high coverage after warm-up windows.
ohlcv_features = [
    "mom1m", "mom6m", "mom12m", "mom36m", "chmom",
    "retvol", "maxret", "idiovol", "std_dolvol",
    "turn", "std_turn", "zerotrade", "dolvol", "baspread",
    "ill", "beta", "betasq", "aeavol", "indmom", "herf"
]

rows = []
n = len(gkx_monthly)
for c in ohlcv_features:
    if c in gkx_monthly.columns:
        nn = int(gkx_monthly[c].notna().sum())
        rows.append({"feature": c, "non_null": nn, "coverage_pct": nn / n * 100})

ohlcv_cov = pd.DataFrame(rows).sort_values("coverage_pct")
display(ohlcv_cov)

## Input Dependency Checks

Many all-NaN GKX variables are accounting-driven and require specific quarterly fields.
The table below maps all-NaN GKX outputs to their required inputs.

In [ ]:
required_inputs = {
    "absacc": ["operating_cash_flow", "net_income", "total_assets"],
    "acc": ["operating_cash_flow", "net_income", "total_assets"],
    "pctacc": ["operating_cash_flow", "net_income"],
    "stdacc": ["operating_cash_flow", "net_income", "total_assets"],
    "stdcf": ["operating_cash_flow"],
    "depr": ["depreciation", "total_assets"],
    "pchdepr": ["depreciation"],
    "dy": ["dividend_yield"],
    "grCAPX": ["capex"],
    "pchcapx_ia": ["capex"],
    "cinvest": ["capex", "total_assets"],
    "chinv": ["inventory"],
    "saleinv": ["inventory", "revenue"],
    "pchsale_pchinvt": ["inventory", "revenue"],
    "pchsaleinv": ["inventory", "revenue"],
    "salerec": ["receivables", "revenue"],
    "pchsale_pchrect": ["receivables", "revenue"],
    "pchsale_pchxsga": ["sga_expense", "revenue"],
    "orgcap": ["sga_expense"],
    "rd": ["research_and_development", "total_assets"],
    "rd_mve": ["research_and_development", "market_cap"],
    "rd_sale": ["research_and_development", "revenue"],
    "realestate": ["real_estate_assets", "total_assets"],
    "tang": ["total_cash", "receivables", "inventory", "ppe", "total_assets"],
    "tb": ["income_tax", "net_income"],
    "secured": ["secured_debt", "total_debt"],
    "chempia": ["employees", "revenue"],
    "hire": ["employees"],
    "roavol": ["roa_or_roaq_history"],
}

rows = []
all_nan_names = set(all_nan["gkx_name"].tolist())

for feat in sorted(all_nan_names):
    deps = required_inputs.get(feat, [])
    for dep in deps:
        rows.append({"gkx_feature": feat, "dependency": dep})

dep_df = pd.DataFrame(rows)
display(dep_df)

In [ ]:
def col_health(df, col):
    if col not in df.columns:
        return {"exists": False, "non_null": 0, "coverage_pct": 0.0}
    nn = int(df[col].notna().sum())
    return {"exists": True, "non_null": nn, "coverage_pct": nn / len(df) * 100}

dependencies = sorted(set(dep_df["dependency"]))

health_rows = []
for dep in dependencies:
    if dep == "roa_or_roaq_history":
        health_rows.append({
            "dependency": dep,
            "final_exists": "roa" in final_df.columns or "roaq" in final_df.columns,
            "final_coverage_pct": None,
            "quarterly_exists": "roa" in quarterly_df.columns,
            "quarterly_coverage_pct": float(quarterly_df["roa"].notna().mean() * 100) if "roa" in quarterly_df.columns else None,
        })
        continue

    f = col_health(final_df, dep)
    q = col_health(quarterly_df, dep)
    health_rows.append({
        "dependency": dep,
        "final_exists": f["exists"],
        "final_coverage_pct": f["coverage_pct"],
        "quarterly_exists": q["exists"],
        "quarterly_coverage_pct": q["coverage_pct"],
    })

health = pd.DataFrame(health_rows).sort_values(["quarterly_exists", "final_exists", "dependency"])
display(health)

In [ ]:
print("Quarterly fundamentals columns available in this run:")
print(sorted(quarterly_df.columns.tolist()))

print("\Count:", len(quarterly_df.columns))
print("Rows:", len(quarterly_df), "| Tickers:", quarterly_df["ticker"].nunique())

In [ ]:
# Valuation merge diagnostics: market_cap/shares_outstanding/industry are ticker-level from yfinance valuation fetch.
for c in ["market_cap", "shares_outstanding", "industry"]:
    if c in final_df.columns:
        by_ticker = final_df.groupby("ticker")[c].apply(lambda s: s.notna().any())
        print(c, "tickers with any value:", int(by_ticker.sum()), "/", len(by_ticker))

## Findings

- OHLCV-derived GKX features are mostly healthy (typically \>96% monthly coverage; lower only from rolling warm-up windows such as 12m/36m momentum and beta history).
- All-NaN GKX features are primarily accounting-driven and depend on columns not present in the stored quarterly fundamentals file for this run.
- The quarterly file for this run has only a limited subset of fields, so downstream formulas for `capex`, `inventory`, `receivables`, `operating_cash_flow`, `depreciation`, `income_tax`, `research_and_development`, `sga_expense`, `employees`, `secured_debt`, `real_estate_assets`, and `ppe` cannot compute.
- `roavol` is all-NaN because it is in `GKX_94` but is not explicitly computed in `build_gkx_characteristics.py`.
- Ticker-level valuation coverage is also sparse in this run, so fields that rely on `market_cap`/`shares_outstanding` are further impacted.